1. 데이터 불러오기

In [1]:
import pandas as pd

데이터 읽어오기 + 혹시 모르니까 원본은 냅두고 .copy()로 바꿔놓기

In [2]:
df_raw = pd.read_csv('../data/processed/food_poisoning_2015_2025.csv', encoding="utf-8-sig")

In [3]:
df = df_raw.copy()

2. 데이터 검증 단계

In [4]:
df.head()

,연도,월,시설,발생건수,환자수
0,2015,1,가정집,3,8
1,2015,1,기타,4,38
2,2015,1,불명,0,0
3,2015,1,음식점,23,127
4,2015,1,학교,1,12


In [5]:
df.tail()

,연도,월,시설,발생건수,환자수
787,2025,12,기타,2,61
788,2025,12,불명,0,0
789,2025,12,음식점,13,402
790,2025,12,학교,3,161
791,2025,12,학교 외 집단급식,2,56


In [6]:
# 데이터 크기 확인    행: 1월 부터 12월까지 총 11년  시설 종류는 6가지 니까  792행, 열: 연도, 월, 시설, 발생건수, 환자수 5열
df.shape

(792, 5)

In [7]:
# 결측치, dtype 확인
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 792 entries, 0 to 791
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   연도      792 non-null    int64
 1   월       792 non-null    int64
 2   시설      792 non-null    str  
 3   발생건수    792 non-null    int64
 4   환자수     792 non-null    int64
dtypes: int64(4), str(1)
memory usage: 31.1 KB


In [8]:
# 결측치 체크
df.isnull().sum()

연도      0
월       0
시설      0
발생건수    0
환자수     0
dtype: int64

In [14]:
# 중복행 있는지 확인
df.duplicated().sum()

np.int64(0)

In [28]:
# 음수 수치 있는지 확인
(df[['발생건수', '환자수']] < 0).any()

발생건수    False
환자수     False
dtype: bool

In [9]:
# 이상치 확인            # 최대지 3298? 너무 갑자기 크게 나오네? 너 두고보자 일단 의심
df[['발생건수', '환자수']].describe().round(2)

,발생건수,환자수
count,792.00,792.00
mean,4.26,93.13
std,6.19,196.68
min,0.00,0.00
25%,0.00,0.00
50%,2.00,32.00
75%,5.00,119.25
max,37.00,3298.00


In [10]:
df['환자수'].max() / df['환자수'].mean()        # 평균의 35배?...

np.float64(35.41284453422633)

In [11]:
df.loc[df['발생건수'] == 37]      # 일단 37이면 봐줄만은 한가? 일단 패스

,연도,월,시설,발생건수,환자수
117,2016,8,음식점,37,167
183,2017,7,음식점,37,265


In [12]:
df.loc[df['환자수'] == 3298]              # 2018년 9월 사건이 있었나?

,연도,월,시설,발생건수,환자수
265,2018,9,기타,12,3298


## 데이터 검증 중 이상값 조사
2018년 9월 전국 각급 학교 급식으로 제공된 초콜릿 케이크를 섭취한 뒤 살모넬라균(살모넬라 톰슨)에 
감염된 집단 식중독 사고가 크게 발생했다.

-참고자료
https://www.yna.co.kr/view/AKR20180907073551017 , 연합뉴스 기사

## raw_data 와 맞는지 검증
같은 *사이트 연도/시설 별 통계, 연도/월 별 통계 다운받아서 제가 합친 데이터와 일치하는 지 검증

data/raw 폴더에 1. (연도별_월별_식중독_통계_.xls)파일, 2. (연도별_원인시설별_식중독_통계_.xls)파일 이랑 비교

*사이트 = 식품의약품안전처에서 운영하는 식품 안전 나라. readme에 출처 적겠습니다. 비교는 눈으로 했습니다.

In [30]:
df.groupby('연도')[['발생건수', '환자수']].sum() # 연별 총 발생건수, 환자 수 데이터 확인 / 일치

,발생건수,환자수
연도,,
2015,330,5981
2016,399,7162
2017,336,5649
2018,363,11504
2019,286,4075
2020,164,2534
2021,245,5160
2022,311,5501
2023,359,8789


In [31]:
df.groupby('월')[['발생건수', '환자수']].sum() # 월별 총 발생건수, 환자 수 데이터 확인 / 일치

,발생건수,환자수
월,,
1,236,2755
2,148,1895
3,217,4613
4,262,5267
5,307,6517
6,342,7408
7,398,9513
8,405,12740
9,364,11901


In [32]:
df.groupby('시설')[['발생건수','환자수']].sum() # 시설 별 총 발생건수, 환자수 데이터 확인  / 일치

,발생건수,환자수
시설,,
가정집,39,181
기타,515,16453
불명,106,1241
음식점,2002,24940
학교,314,17933
학교 외 집단급식,401,13011
